# Part 5: Cleanup, Validation & Save

Sections 5A-5F: Smelter name standardization, deduplication, missing-edge repair, validation, diagnostics, and final file export.

**Depends on:** Part 4 (`_pipeline_state_4.pkl`)

In [ ]:
import os, shutil, re, pickle
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
DIR_PRELIM = os.path.join(BASE_DIR, "Preliminary")
COCHILCO_PATH = os.path.join(BASE_DIR, "data", "COCHILCO_Production_2005_2024.xlsx")

_cochilco_orig_candidates = [
    os.path.join(BASE_DIR, "data", "1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    os.path.join(BASE_DIR, "data", "Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    "/Users/leoss/Downloads/1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
    "/Users/leoss/Downloads/Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
]
COCHILCO_ORIG = next((p for p in _cochilco_orig_candidates if os.path.exists(p)), _cochilco_orig_candidates[0])

SALIDAS_PATH = os.path.join(BASE_DIR, "data", "salidas_2024_clean.csv")

# ── Load state from Part 4 ─────────────────────────────────────────────────
_state_path = os.path.join(DIR_PRELIM, "_pipeline_state_4.pkl")
with open(_state_path, "rb") as _f:
    _state = pickle.load(_f)

inv   = _state["inv"]
links = _state["links"]
edges = _state["edges"]

# ports_df: Part 2 built this locally from PORTS but never saved it.
PORTS    = _state.get("PORTS", [])
ports_df = pd.DataFrame(PORTS)

# File paths: derived from DIR_PRELIM (same naming convention used throughout).
inv_path   = os.path.join(DIR_PRELIM, "Chile_Minerals_Inventory.csv")
links_path = os.path.join(DIR_PRELIM, "Chile_Mine_Plant_Links.csv")

# export_df: saved by Part 3 into state_4.
export_df = _state.get("export_df", pd.DataFrame())

# cu_total: national Cu production total, read in Part 1 but never serialised.
# Re-read from COCHILCO to keep the validation denominator accurate.
if os.path.exists(COCHILCO_PATH):
    _nat = pd.read_excel(COCHILCO_PATH, sheet_name="A_National_Production", header=3, index_col=0)
    _nat.columns = [int(c) if isinstance(c, (int, float)) else c for c in _nat.columns]
    _latest_yr = max(c for c in _nat.columns if isinstance(c, int))
    _cu_row = next(
        (r for r in _nat.index if re.search(r"COBRE.*Miles de TM", str(r), re.IGNORECASE)), None
    )
    cu_total = float(_nat.loc[_cu_row, _latest_yr]) if _cu_row else inv["COCHILCO_CU_2024_KMT"].sum()
    print(f"  Cu national total ({_latest_yr}): {cu_total:,.1f} kMT")
else:
    cu_total = inv["COCHILCO_CU_2024_KMT"].sum()
    print(f"  Warning: COCHILCO_PATH not found; cu_total set to matched total ({cu_total:,.1f} kMT)")

n_idle_links = _state.get("n_idle_links", 0)

comm_col             = _state.get("comm_col", "COMMODITY_LIST_STR")
idle_mines           = _state.get("idle_mines", set())
COMPANY_TO_DEPOSIT   = _state.get("COMPANY_TO_DEPOSIT", {})
CODELCO_EXTRA_SEARCH = _state.get("CODELCO_EXTRA_SEARCH", {})
SMELTERS             = _state.get("SMELTERS", [])
SMELTER_NAME_MAP     = _state.get("SMELTER_NAME_MAP", {})
DEDICATED_PORT       = _state.get("DEDICATED_PORT", {})
CODELCO_CATHODE_ROUTING = _state.get("CODELCO_CATHODE_ROUTING", {})

print(f"Loaded state from Part 4: {len(inv)} inv rows, {len(links)} link rows, {len(edges)} edge rows")
print(f"Ports: {len(ports_df)}  |  Export edges: {len(export_df)}")

# ── Shared utility functions ───────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def parse_comm_list(val):
    if pd.isna(val): return []
    return [x.strip() for x in str(val).split(",") if x.strip()]

def add_commodity(row_idx, commodity, df, col):
    current = parse_comm_list(df.at[row_idx, col])
    if commodity not in current:
        current.append(commodity)
        df.at[row_idx, col] = ", ".join(current)
        return True
    return False

def nearest_port(lat, lon, product_type="concentrate"):
    best_dist, best_port = float("inf"), None
    for port in PORTS:
        if product_type == "cathode" and "cathode" not in port["products"].lower():
            continue
        if product_type == "concentrate" and "concentrate" not in port["products"].lower():
            continue
        dist = haversine_km(lat, lon, port["lat"], port["lon"])
        if dist < best_dist:
            best_dist, best_port = dist, port
    return best_port, best_dist

def section_header(title, width=65):
    print(f"\n{'=' * width}\n{title}\n{'=' * width}")

def search_inventory(inv_df, terms, require_mine=False):
    matched = set()
    name_lower = inv_df["FACILITY_NAME"].str.lower().str.strip()
    for term in terms:
        mask = name_lower.str.contains(term.lower(), na=False, regex=False)
        if require_mine:
            mask = mask & inv_df["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matched.update(inv_df[mask].index)
    return list(matched)


In [ ]:

# ── 5A. Smelter name standardization ──────────────────────────────────────

section_header("5A. SMELTER NAME STANDARDIZATION")

renamed_count = 0
for canonical, inv_name in SMELTER_NAME_MAP.items():
    exists = inv["FACILITY_NAME"].eq(inv_name).any()
    print(f"  {canonical:<35} -> {inv_name:<55} {'OK' if exists else 'NOT FOUND'}")
    if canonical != inv_name:
        for col in ["FROM_NAME", "TO_NAME"]:
            mask = edges[col] == canonical
            if mask.any():
                edges.loc[mask, col] = inv_name
                renamed_count += mask.sum()
print(f"\nTotal renames: {renamed_count}")

# Handle Las Ventanas / Ventanas ambiguity
las_v   = inv[inv["FACILITY_NAME"].str.contains("Las Ventanas", case=False, na=False)]
v_plain = inv[inv["FACILITY_NAME"].str.contains("Ventanas refinery", case=False, na=False) &
              ~inv["FACILITY_NAME"].str.contains("Las Ventanas", case=False, na=False)]
if len(las_v) > 0 and len(v_plain) > 0:
    print(f"\n  Note: Both '{las_v.iloc[0]['FACILITY_NAME']}' AND '{v_plain.iloc[0]['FACILITY_NAME']}' exist.")

# ── 5B. Andacollo Oro mine link ────────────────────────────────────────────

section_header("5B. ANDACOLLO ORO MINE")

andacollo = inv[inv["FACILITY_NAME"].str.contains("Andacollo", case=False, na=False)]
for _, row in andacollo.iterrows():
    prod = row.get("COCHILCO_CU_2024_KMT", np.nan)
    prod_str = f"{prod:.1f} kMT" if pd.notna(prod) else ""
    print(f"  {row['FACILITY_NAME']:<45} {row['FACILITY_TYPE']:<20} {prod_str}")

existing_anda = links[links["MINE_NAME"].str.contains("Andacollo", case=False, na=False)]
if len(existing_anda) > 0:
    print(f"  Link already exists: {existing_anda.iloc[0]['MINE_NAME']} -> {existing_anda.iloc[0]['PLANT_NAME']}")
else:
    anda_mine = inv[inv["FACILITY_NAME"].str.contains("Andacollo", case=False, na=False) &
                    inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]
    if len(anda_mine) > 0:
        mine_row = anda_mine.iloc[0]
        if pd.notna(mine_row["LATITUD"]):
            sxew_plants = inv[(inv["CHAIN_STAGE"] == "sx_ew") & inv["LATITUD"].notna()].copy()
            sxew_plants["_dist"] = sxew_plants.apply(
                lambda r: haversine_km(mine_row["LATITUD"], mine_row["LONGITUD"], r["LATITUD"], r["LONGITUD"]), axis=1)
            nearby = sxew_plants[sxew_plants["_dist"] < 50].sort_values("_dist")
            if len(nearby) > 0:
                target = nearby.iloc[0]
                new_link = {"MINE_NAME": mine_row["FACILITY_NAME"], "PLANT_NAME": target["FACILITY_NAME"],
                            "MINE_LAT": mine_row["LATITUD"], "MINE_LON": mine_row["LONGITUD"],
                            "PLANT_LAT": target["LATITUD"], "PLANT_LON": target["LONGITUD"],
                            "SHARED_COMMODITIES": "Copper", "DISTANCE_KM": round(target["_dist"], 1),
                            "PRODUCT_FORM": "cathode_sxew"}
                links = pd.concat([links, pd.DataFrame([new_link])], ignore_index=True)
                edges = pd.concat([edges, pd.DataFrame([{
                    "FROM_NAME": mine_row["FACILITY_NAME"], "FROM_TYPE": "mine",
                    "FROM_LAT": mine_row["LATITUD"], "FROM_LON": mine_row["LONGITUD"],
                    "TO_NAME": target["FACILITY_NAME"], "TO_TYPE": "plant",
                    "TO_LAT": target["LATITUD"], "TO_LON": target["LONGITUD"],
                    "EDGE_TYPE": "mine_to_plant", "PRODUCT_FORM": "cathode_sxew",
                    "COMMODITIES": "Copper", "DISTANCE_KM": round(target["_dist"], 1),
                }])], ignore_index=True)
                print(f"  -> Created link + edge: {mine_row['FACILITY_NAME']} -> {target['FACILITY_NAME']}")

# ── 5B.2: Missing smelter-to-port edges ───────────────────────────────────
#
# Root cause: smelter_inv_map in Part 2 resolved canonical smelter names to
# SHORT inventory names via keyword search. The LINKS table, however, records
# the LONG facility name as found in the source data. This creates a split:
# mine-to-plant edges land on the long name, while smelter-to-port edges were
# built from the short search-matched name. The three smelters below ended up
# with m2p edges on their long names but zero s2p edges.
#
# Fix: add s2p edges from the long names using the SMELTERS export_port config.

section_header("5B.2: MISSING SMELTER-TO-PORT EDGES")

# Map long inventory name -> list of export ports + product + coordinates.
# Sourced from the SMELTERS definitions in Part 0.
MISSING_S2P = [
    {
        "from_name": "Caletones smelter (anodes). refinery (fire-refined ingots), and SX-EW plant",
        "ports": ["San Antonio", "Ventanas"],
        "lat": -34.12, "lon": -70.48,
        "product": "cathode",
    },
    {
        "from_name": "Hernán Videla Lira smelter (anodes and blister)",
        "ports": ["Barquito", "Caldera"],
        "lat": -27.37, "lon": -70.30,
        "product": "blister",
    },
    {
        # Las Ventanas is a Codelco refinery that processes anodes from
        # several smelters and exports cathode through the Ventanas port terminal.
        "from_name": "Las Ventanas refinery and smelter",
        "ports": ["Ventanas", "San Antonio"],
        "lat": -32.74, "lon": -71.49,
        "product": "cathode",
    },
]

_added_s2p = 0
for entry in MISSING_S2P:
    existing = edges[
        (edges["EDGE_TYPE"] == "smelter_to_port") &
        (edges["FROM_NAME"] == entry["from_name"])
    ]
    if len(existing) > 0:
        print(f"  {entry['from_name'][:55]} already has {len(existing)} s2p edges, skipping")
        continue
    for port_name in entry["ports"]:
        port = next((p for p in PORTS if p["name"] == port_name), None)
        if not port:
            print(f"  WARNING: port '{port_name}' not in PORTS list")
            continue
        new_edge = {
            "FROM_NAME": entry["from_name"], "FROM_TYPE": "smelter",
            "FROM_LAT": entry["lat"], "FROM_LON": entry["lon"],
            "TO_NAME": port["name"], "TO_TYPE": "port",
            "TO_LAT": port["lat"], "TO_LON": port["lon"],
            "EDGE_TYPE": "smelter_to_port",
            "PRODUCT_FORM": entry["product"],
            "COMMODITIES": "Copper",
            "DISTANCE_KM": round(haversine_km(entry["lat"], entry["lon"], port["lat"], port["lon"]), 1),
        }
        edges = pd.concat([edges, pd.DataFrame([new_edge])], ignore_index=True)
        _added_s2p += 1
        print(f"  Added: {entry['from_name'][:50]} -> {port_name}")

print(f"\n  Total new smelter-to-port edges: {_added_s2p}")

# ── 5B.3: Caserones mine-to-plant link ────────────────────────────────────
#
# Caserones (124.6 kMT) had no mine-to-plant edge, leaving it disconnected in
# path traceability. The mine has its own onsite concentrator. Try to match by
# name first; fall back to nearest concentrator within 50 km; if no concentrator
# is found, add a direct mine-to-concentrate-port edge.

section_header("5B.3: CASERONES MINE-TO-PLANT LINK")

existing_caserones = links[links["MINE_NAME"].str.contains("Caserones", case=False, na=False)]
if len(existing_caserones) > 0:
    print(f"  Link already exists: {existing_caserones.iloc[0]['MINE_NAME']} -> {existing_caserones.iloc[0]['PLANT_NAME']}")
else:
    caserones_mine = inv[
        inv["FACILITY_NAME"].str.contains("Caserones", case=False, na=False) &
        inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
    ]
    if len(caserones_mine) == 0:
        print("  No Caserones mine record found in inventory")
    else:
        mine_row = caserones_mine.iloc[0]
        print(f"  Mine: {mine_row['FACILITY_NAME']}  coords: ({mine_row['LATITUD']}, {mine_row['LONGITUD']})")
        if pd.isna(mine_row["LATITUD"]):
            print("  Mine has no coordinates — cannot auto-link")
        else:
            concentrators = inv[(inv["CHAIN_STAGE"] == "concentration") & inv["LATITUD"].notna()].copy()
            concentrators["_dist"] = concentrators.apply(
                lambda r: haversine_km(mine_row["LATITUD"], mine_row["LONGITUD"], r["LATITUD"], r["LONGITUD"]), axis=1)
            # Prefer a concentrator explicitly named Caserones
            named = concentrators[concentrators["FACILITY_NAME"].str.contains("Caserones", case=False, na=False)]
            if len(named) > 0:
                target = named.sort_values("_dist").iloc[0]
                print(f"  Matched by name: {target['FACILITY_NAME']}  ({target['_dist']:.1f} km)")
            else:
                nearby = concentrators[concentrators["_dist"] < 50].sort_values("_dist")
                target = nearby.iloc[0] if len(nearby) > 0 else None
                if target is not None:
                    print(f"  Matched by proximity: {target['FACILITY_NAME']}  ({target['_dist']:.1f} km)")

            if target is not None:
                dist_km = round(float(target["_dist"]), 1)
                new_link = {
                    "MINE_NAME": mine_row["FACILITY_NAME"], "PLANT_NAME": target["FACILITY_NAME"],
                    "MINE_LAT": mine_row["LATITUD"], "MINE_LON": mine_row["LONGITUD"],
                    "PLANT_LAT": target["LATITUD"], "PLANT_LON": target["LONGITUD"],
                    "SHARED_COMMODITIES": "Copper", "DISTANCE_KM": dist_km,
                    "PRODUCT_FORM": "concentrate",
                }
                links = pd.concat([links, pd.DataFrame([new_link])], ignore_index=True)
                edges = pd.concat([edges, pd.DataFrame([{
                    "FROM_NAME": mine_row["FACILITY_NAME"], "FROM_TYPE": "mine",
                    "FROM_LAT": mine_row["LATITUD"], "FROM_LON": mine_row["LONGITUD"],
                    "TO_NAME": target["FACILITY_NAME"], "TO_TYPE": "plant",
                    "TO_LAT": target["LATITUD"], "TO_LON": target["LONGITUD"],
                    "EDGE_TYPE": "mine_to_plant", "PRODUCT_FORM": "concentrate",
                    "COMMODITIES": "Copper", "DISTANCE_KM": dist_km,
                }])], ignore_index=True)
                print(f"  -> Added link + edge: {mine_row['FACILITY_NAME']} -> {target['FACILITY_NAME']}")
            else:
                # No concentrator found: add direct mine-to-port edge so the
                # 124.6 kMT is not silently excluded from traceability.
                port, dist = nearest_port(mine_row["LATITUD"], mine_row["LONGITUD"], "concentrate")
                if port:
                    edges = pd.concat([edges, pd.DataFrame([{
                        "FROM_NAME": mine_row["FACILITY_NAME"], "FROM_TYPE": "mine",
                        "FROM_LAT": mine_row["LATITUD"], "FROM_LON": mine_row["LONGITUD"],
                        "TO_NAME": port["name"], "TO_TYPE": "port",
                        "TO_LAT": port["lat"], "TO_LON": port["lon"],
                        "EDGE_TYPE": "concentrate_to_port", "PRODUCT_FORM": "concentrate",
                        "COMMODITIES": "Copper", "DISTANCE_KM": round(dist, 1),
                    }])], ignore_index=True)
                    print(f"  -> No concentrator found; added direct mine-to-port edge -> {port['name']}")

# ── 5C. Deduplication ─────────────────────────────────────────────────────

section_header("5C. DEDUPLICATION")

before = len(edges)
edges = edges.drop_duplicates(
    subset=["FROM_NAME", "TO_NAME", "EDGE_TYPE", "COMMODITIES", "PRODUCT_FORM"], keep="first"
)
print(f"  Removed {before - len(edges)} duplicate edges ({before} -> {len(edges)})")

# ── 5D. Validation ────────────────────────────────────────────────────────

section_header("5D. VALIDATION")

issues = []
cu_matched_total = inv["COCHILCO_CU_2024_KMT"].sum()
cu_count = inv["COCHILCO_CU_2024_KMT"].notna().sum()
print(f"  Cu production: {cu_count} records, {cu_matched_total:,.1f} / {cu_total:,.1f} kMT "
      f"({cu_matched_total/cu_total*100:.1f}%)")

for _mineral_name, _col_name, _unit_label in [
    ("Mo", "COCHILCO_MO_2024_MT", "MT"),
    ("Au", "COCHILCO_AU_2024_KG", "Kg"),
    ("Ag", "COCHILCO_AG_2024_KG", "Kg"),
    ("Fe", "COCHILCO_FE_2024_KMT", "kMT"),
    ("Zn", "COCHILCO_ZN_2024_MT", "MT"),
]:
    if _col_name in inv.columns and inv[_col_name].notna().any():
        _n = inv[_col_name].notna().sum()
        _total = inv[_col_name].sum()
        print(f"  {_mineral_name} production: {_n} records, {_total:,.1f} {_unit_label}")

# Path traceability — set-based (no nested loop)
# Build lookup sets once to avoid repeated per-row DataFrame scans.
_port_sources = set(edges.loc[
    edges["EDGE_TYPE"].isin(["concentrate_to_port", "sxew_to_port", "smelter_to_port"]), "FROM_NAME"
])
_c2s = edges[edges["EDGE_TYPE"] == "concentrate_to_smelter"][["FROM_NAME", "TO_NAME"]]
_smelter_port_sources = set(edges.loc[edges["EDGE_TYPE"] == "smelter_to_port", "FROM_NAME"])
_via_smelter = set(_c2s.loc[_c2s["TO_NAME"].isin(_smelter_port_sources), "FROM_NAME"])
_plants_reach_port = _port_sources | _via_smelter

_m2p = edges[edges["EDGE_TYPE"] == "mine_to_plant"][["FROM_NAME", "TO_NAME"]]
_mine_reaches = (
    _m2p.groupby("FROM_NAME")["TO_NAME"]
    .apply(set)
    .reset_index()
    .rename(columns={"FROM_NAME": "MINE", "TO_NAME": "PLANTS"})
)
_mine_reaches["reaches_port"] = _mine_reaches["PLANTS"].apply(
    lambda plants: bool(plants & _plants_reach_port)
)
_connected_mines = set(_mine_reaches.loc[_mine_reaches["reaches_port"], "MINE"])

# FIX: some facilities have COCHILCO production attached directly to the SX-EW
# or concentrator record (e.g. Zaldivar SX-EW plant, Tres Valles SX-EW plant)
# rather than to a separate mine record. These don't appear as mine_to_plant
# FROM_NAMEs, so they would fall through to the disconnected list. Check whether
# the producer itself appears in sxew_to_port or concentrate_to_port FROM_NAMEs.
_direct_to_port = set(edges.loc[
    edges["EDGE_TYPE"].isin(["sxew_to_port", "concentrate_to_port"]), "FROM_NAME"
])

cu_producers = inv[(inv["COCHILCO_CU_2024_KMT"].notna()) & (inv["COCHILCO_CU_2024_KMT"] > 0)]
connected_prod, disconnected = 0.0, []

for _, mrow in cu_producers.iterrows():
    mine_name = mrow["FACILITY_NAME"]
    is_connected = (
        mine_name in _connected_mines
        or any(m.startswith(mine_name[:8]) for m in _connected_mines)
        or mine_name in _direct_to_port  # direct sx_ew / concentrator producers
    )
    if is_connected:
        connected_prod += mrow["COCHILCO_CU_2024_KMT"]
    else:
        disconnected.append((mine_name, mrow["COCHILCO_CU_2024_KMT"]))

print(f"\n  Path traceability:")
print(f"    Mines reaching port: {cu_count - len(disconnected)} / {cu_count}")
print(f"    Production reaching ports: {connected_prod:,.1f} / {cu_matched_total:,.1f} kMT "
      f"({connected_prod/cu_matched_total*100:.1f}%)")
if disconnected:
    for name, prod in sorted(disconnected, key=lambda x: -x[1]):
        print(f"      {name:<45} {prod:>8.1f} kMT")
    issues.append(f"{len(disconnected)} mines ({sum(p for _,p in disconnected):,.1f} kMT) don't reach a port")

# ── 5E. Compact diagnostics ───────────────────────────────────────────────

section_header("DIAGNOSTICS SUMMARY")

for etype in edges["EDGE_TYPE"].unique():
    n = len(edges[edges["EDGE_TYPE"] == etype])
    print(f"  {etype:<25} {n:>5} edges")

print(f"\n  Downstream coverage by commodity:")
downstream_check = edges[edges["EDGE_TYPE"] != "mine_to_plant"]
if "COMMODITIES" in downstream_check.columns:
    for comm, count in downstream_check["COMMODITIES"].value_counts().items():
        print(f"    {comm:<20} {count:>5} downstream edges")

print(f"\n  Smelter connectivity:")
for name in sorted(inv[inv["CHAIN_STAGE"] == "smelting"]["FACILITY_NAME"].unique()):
    m2p = len(edges[(edges["EDGE_TYPE"] == "mine_to_plant") & (edges["TO_NAME"] == name)])
    s2p = len(edges[(edges["EDGE_TYPE"] == "smelter_to_port") & (edges["FROM_NAME"] == name)])
    conn = f"m2p={m2p}, s2p={s2p}" if (m2p + s2p) > 0 else "DISCONNECTED"
    print(f"    {name:<60} [{conn}]")

print(f"\nNote: {len(idle_mines)} idle mines retained in inventory, {n_idle_links} phantom links removed.")

# ── 5F. Save all files ────────────────────────────────────────────────────

section_header("5F. SAVE")

edges = edges.copy()  # avoid SettingWithCopyWarning on in-place sort
edges.sort_values(["EDGE_TYPE", "FROM_NAME", "TO_NAME"], inplace=True)
edges.reset_index(drop=True, inplace=True)

for et, count in edges["EDGE_TYPE"].value_counts().sort_index().items():
    print(f"  {et:<25} {count:>5}")
print(f"  {'TOTAL':<25} {len(edges):>5}")

if issues:
    print(f"\nRemaining issues:")
    for i, iss in enumerate(issues, 1):
        print(f"  {i}. {iss}")

ds = edges[edges["EDGE_TYPE"] != "mine_to_plant"]
export_df.to_csv(os.path.join(DIR_PRELIM, "Chile_Export_Destinations.csv"), index=False)
edges.to_csv(os.path.join(DIR_PRELIM, "Chile_Supply_Chain_Edges.csv"), index=False)
inv.to_csv(inv_path, index=False)
links.to_csv(links_path, index=False)
ds.to_csv(os.path.join(DIR_PRELIM, "Chile_Downstream_Links.csv"), index=False)
ports_df.to_csv(os.path.join(DIR_PRELIM, "Chile_Ports.csv"), index=False)

print(f"\nSaved:")
for name, n in [("Chile_Minerals_Inventory.csv", len(inv)), ("Chile_Mine_Plant_Links.csv", len(links)),
                ("Chile_Supply_Chain_Edges.csv", len(edges)), ("Chile_Downstream_Links.csv", len(ds)),
                ("Chile_Export_Destinations.csv", len(export_df)), ("Chile_Ports.csv", len(ports_df))]:
    print(f"  {name:<35} ({n} records)")

# ── Serialise forward for Part 5 (distance analysis) ──────────────────────
_state_out = dict(_state)
_state_out.update({
    "inv": inv,
    "links": links,
    "edges": edges,
    "export_df": export_df,
    "ports_df": ports_df,
    "inv_path": inv_path,
    "links_path": links_path,
    "cu_total": cu_total,
    "n_idle_links": n_idle_links,
})
_out_path = os.path.join(DIR_PRELIM, "_pipeline_state_5.pkl")
with open(_out_path, "wb") as _f:
    pickle.dump(_state_out, _f)
print(f"\nState serialised -> {_out_path}")
